In [0]:
# Confluent Cloud Kafka Credentials & Settings
kafka_bootstrap_servers = "pkc-7prvp.centralindia.azure.confluent.cloud:9092"  # Apna Confluent bootstrap server URL dalein
kafka_api_key = dbutils.secrets.get("atliq", "kafka_api_key")                    # Apni Confluent API key dalein
kafka_api_secret = dbutils.secrets.get("atliq", "kafka_api_secret")                # Apna Confluent API secret dalein
kafka_topic = "atliq.orders.events"                 # Topic name

display(spark.sql("SELECT json_payload FROM atliq.streaming.bronze_order_events LIMIT 5"))

json_payload
"{""event_id"": ""bd3ebcf2-1e09-4049-834e-b4b9c9bc638e"", ""event_type"": ""order_placed"", ""event_ts"": ""2026-10-28T16:39:09.594Z"", ""order_id"": 100008, ""customer_id"": 71, ""city"": ""Bengaluru"", ""product_id"": 42, ""quantity"": 1, ""order_amount"": 449, ""payment_method"": null}"
"{""event_id"": ""6a35c8dd-d8eb-482b-9efa-86dd0230ff5f"", ""event_type"": ""payment_received"", ""event_ts"": ""2026-10-28T16:39:09.594Z"", ""order_id"": 100008, ""customer_id"": 71, ""city"": ""Bengaluru"", ""product_id"": 42, ""quantity"": 1, ""order_amount"": 449, ""payment_method"": ""Debit Card""}"
"{""event_id"": ""3dc7e493-01a5-43ff-a084-8d531c476eb4"", ""event_type"": ""order_cancelled"", ""event_ts"": ""2026-10-28T16:39:11.600Z"", ""order_id"": 100008, ""customer_id"": 71, ""city"": ""Bengaluru"", ""product_id"": 42, ""quantity"": 1, ""order_amount"": 449, ""payment_method"": null}"
"{""event_id"": ""d3f09716-12b9-4069-b155-68f5a88387ed"", ""event_type"": ""order_placed"", ""event_ts"": ""2026-10-28T16:39:13.604Z"", ""order_id"": 100014, ""customer_id"": 173, ""city"": ""Jaipur"", ""product_id"": 24, ""quantity"": 3, ""order_amount"": 2397, ""payment_method"": null}"
"{""event_id"": ""97564366-9fa3-4658-adcb-91a227393424"", ""event_type"": ""payment_received"", ""event_ts"": ""2026-10-28T16:39:13.604Z"", ""order_id"": 100014, ""customer_id"": 173, ""city"": ""Jaipur"", ""product_id"": 24, ""quantity"": 3, ""order_amount"": 2397, ""payment_method"": ""Wallet""}"


In [0]:
for q in spark.streams.active:
    q.stop()
 
for t in ["bronze_order_events", "silver_order_events",
          "gold_revenue_5min", "gold_realtime_kpis"]:
    spark.sql(f"DROP TABLE IF EXISTS atliq.streaming.{t}")
 
for c in ["bronze", "silver", "gold", "gold_5min"]:
    dbutils.fs.rm(f"/Volumes/atliq/streaming/checkpoints/{c}", True)
 
spark.sql("CREATE CATALOG IF NOT EXISTS atliq")
spark.sql("CREATE SCHEMA IF NOT EXISTS atliq.streaming")
spark.sql("CREATE VOLUME IF NOT EXISTS atliq.streaming.checkpoints")

DataFrame[]

In [0]:
from pyspark.sql import functions as F
 
df_kafka_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", kafka_bootstrap_servers)
    .option("subscribe", kafka_topic)
    .option("startingOffsets", "earliest")   # rebuild from events already in the topic
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config",
            f"kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required "
            f"username='{kafka_api_key}' password='{kafka_api_secret}';")
    .load()
)
 
df_bronze = df_kafka_raw.select(
    F.col("key").cast("string").alias("event_key"),
    F.col("value").cast("string").alias("json_payload"),
    F.col("topic"),
    F.col("partition"),
    F.col("offset"),
    F.col("timestamp").alias("kafka_received_at"),
)
 
bronze_query = (
    df_bronze.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation", "/Volumes/atliq/streaming/checkpoints/bronze")
    .toTable("atliq.streaming.bronze_order_events")
)
bronze_query.awaitTermination()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StringType, DoubleType
 
order_schema = (
    StructType()
    .add("event_id", StringType())
    .add("event_type", StringType())      # NOT "status" - guide and Gold use event_type
    .add("order_id", StringType())
    .add("customer_id", StringType())
    .add("product_id", StringType())
    .add("order_amount", DoubleType())    # 'N/A' becomes NULL
    .add("event_ts", StringType())        # converted to timestamp below
    # add any other keys you saw in CELL 1 (e.g. city, payment_method)
)
 
df_parsed = (
    spark.readStream.table("atliq.streaming.bronze_order_events")
    .select(F.from_json("json_payload", order_schema).alias("d"), "kafka_received_at")
    .select("d.*", "kafka_received_at")
    .withColumn("event_ts", F.to_timestamp("event_ts"))
)
 
df_silver = (
    df_parsed
    .filter(F.col("event_id").isNotNull()
            & F.col("event_ts").isNotNull()
            & F.col("order_amount").isNotNull())      # drop broken records first
    .withWatermark("event_ts", "10 minutes")
    .dropDuplicates(["event_id", "event_ts"])
)
 
silver_query = (
    df_silver.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation", "/Volumes/atliq/streaming/checkpoints/silver")
    .toTable("atliq.streaming.silver_order_events")
)
silver_query.awaitTermination()

In [0]:
display(spark.sql("""
    SELECT event_type, COUNT(*) AS n
    FROM atliq.streaming.silver_order_events
    GROUP BY event_type
"""))

event_type,n
order_cancelled,130
order_placed,694
payment_received,583
order_shipped,481


In [0]:
from pyspark.sql import functions as F
 
df_gold = (
    spark.readStream.table("atliq.streaming.silver_order_events")
    .filter(F.col("event_type") == "payment_received")
    .withWatermark("event_ts", "10 minutes")
    .groupBy(F.window("event_ts", "5 minutes"))
    .agg(
        F.count("*").alias("orders_paid"),
        F.sum("order_amount").alias("revenue"),
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "orders_paid",
        "revenue",
    )
)
 
gold_query = (
    df_gold.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation", "/Volumes/atliq/streaming/checkpoints/gold_5min")
    .toTable("atliq.streaming.gold_revenue_5min")
)
gold_query.awaitTermination()

In [0]:
display(spark.sql("SELECT * FROM atliq.streaming.gold_revenue_5min ORDER BY window_start DESC"))

window_start,window_end,orders_paid,revenue
2026-10-28T19:10:00.000Z,2026-10-28T19:15:00.000Z,5,11287.0
2026-10-28T19:05:00.000Z,2026-10-28T19:10:00.000Z,1,2598.0
2026-10-28T19:00:00.000Z,2026-10-28T19:05:00.000Z,1,4598.0
2026-10-28T18:55:00.000Z,2026-10-28T19:00:00.000Z,3,21092.0
2026-10-28T18:50:00.000Z,2026-10-28T18:55:00.000Z,2,7895.0
2026-10-28T18:05:00.000Z,2026-10-28T18:10:00.000Z,19,45460.0
2026-10-28T18:00:00.000Z,2026-10-28T18:05:00.000Z,11,44183.0
2026-10-28T16:35:00.000Z,2026-10-28T16:40:00.000Z,22,63506.0
